# Planning

**Prerequisites:** `03_retries_and_reflection.ipynb`, and from `01_foundations`: `05_structured_outputs.ipynb`, `08_asyncio.ipynb`

ReAct decides one step at a time. Every turn, the whole conversation goes back to the model and it works out what to do next from scratch. For a two-step task that's fine. For a six-step task it's six full-context calls to answer a question whose shape was obvious from the start.

Deciding the whole sequence up front changes that, and it changes something more interesting too. A list of steps written in advance is a *structure* — you can look at it before running any of it, and see that step 1 needs step 0's result while step 2 needs nothing from either. ReAct can never see that, because it only ever knows about the step it's on.

This notebook writes plans of two different kinds and executes them.

**By the end you'll have:**
- Built `PlannerThink`, producing an entire validated sequence of tool calls in one structured call
- Solved the problem that makes planning hard: writing an argument whose value won't exist until an earlier step runs, via `$stepN` and `$stepN.field` references
- Seen that those references *are* the dependency graph, and watched `Graph` start every step the moment its own references are satisfied — with a measured wall-clock win
- Run the full compiler loop, plan → graph → replan, and seen why "deciding to answer" has to be an Act like any other
- Built the other kind of plan: one made of **goals**, which names no tool, emits no decision, and leaves the how to a `Think`


## Setup

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))

import json
import time

from chat import Conversation
from runtime import Runtime, Graph
from observe import Observe
from think import Think, PlannerThink, step_dependencies, resolve_args
from act import Act

Three tools this time. `get_weather` is new and matters for two reasons: it returns a **JSON object with named fields** rather than a single value, and — along with `get_current_time` — it sleeps for a moment, so that running things at the same time is something you can measure rather than take on faith.

In [2]:
from datetime import datetime
from typing import Literal
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from pydantic import BaseModel, field_validator

WEATHER = {"Tokyo": {"temp_f": 88.0, "condition": "clear"},
           "London": {"temp_f": 61.0, "condition": "overcast"}}

def get_weather(city):
    time.sleep(1.5)                      # stand-in for a real network call
    return json.dumps(WEATHER[city])

def get_current_time(timezone):
    time.sleep(1.5)
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M %Z")

def convert_temperature(value, from_unit, to_unit):
    if from_unit.upper() == "F" and to_unit.upper() == "C":
        return round((value - 32) * 5 / 9, 1)
    if from_unit.upper() == "C" and to_unit.upper() == "F":
        return round(value * 9 / 5 + 32, 1)
    raise ValueError(f"unsupported conversion: {from_unit} to {to_unit}")

class GetWeatherArgs(BaseModel):
    city: Literal["Tokyo", "London"]

class GetCurrentTimeArgs(BaseModel):
    timezone: str

    @field_validator("timezone")
    @classmethod
    def timezone_must_be_valid(cls, v):
        try:
            ZoneInfo(v)
        except ZoneInfoNotFoundError:
            raise ValueError(f"'{v}' is not a valid IANA timezone")
        return v

class ConvertTemperatureArgs(BaseModel):
    value: float
    from_unit: Literal["C", "F"]
    to_unit: Literal["C", "F"]

FUNCTIONS = {"get_weather": get_weather, "get_current_time": get_current_time,
             "convert_temperature": convert_temperature}
SCHEMAS = {"get_weather": GetWeatherArgs, "get_current_time": GetCurrentTimeArgs,
           "convert_temperature": ConvertTemperatureArgs}

tools = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get the current weather in a city.",
        "parameters": {"type": "object",
                       "properties": {"city": {"type": "string", "enum": ["Tokyo", "London"]}},
                       "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a given IANA timezone.",
        "parameters": {"type": "object",
                       "properties": {"timezone": {"type": "string", "description": "IANA timezone name"}},
                       "required": ["timezone"]}}},
    {"type": "function", "function": {
        "name": "convert_temperature",
        "description": "Convert a temperature value between Celsius and Fahrenheit.",
        "parameters": {"type": "object",
                       "properties": {"value": {"type": "number"},
                                      "from_unit": {"type": "string", "enum": ["C", "F"]},
                                      "to_unit": {"type": "string", "enum": ["C", "F"]}},
                       "required": ["value", "from_unit", "to_unit"]}}},
]

# what each tool returns -- the planner needs this to reference a field of a result it hasn't seen
OUTPUTS = {
    "get_weather": 'a JSON object with fields "temp_f" (number, degrees Fahrenheit) and "condition" (string)',
    "get_current_time": "a single formatted date-time string",
    "convert_temperature": "a single number, the converted temperature",
}

REQUEST = "What's the current time in Tokyo, and what's the temperature there in Celsius?"

## The argument that doesn't exist yet

Planning runs into one problem immediately, and everything else about `PlannerThink` is shaped by it.

To answer the request, something has to convert Tokyo's temperature to Celsius. But the planner is writing the plan *before* anything runs — it has no idea what `get_weather` will return. It cannot write `{"value": 88.0, ...}`, because 88.0 is not knowable yet. And it must not guess, because a plausible guess is worse than an obvious placeholder.

So the plan gets a way to say "whatever step 1 returns":

- `"$step0"` — the entire result of step 0
- `"$step1.temp_f"` — just the `temp_f` field, when that step returns a JSON object

This is why `PlannerThink` takes an `outputs` dict. To reference `.temp_f` the planner has to know `get_weather` returns something with a `temp_f` in it, and no tool *schema* describes a return value — schemas describe arguments. `_tool_descriptions()` stitches the two together:

```python
def _tool_descriptions(self):
    return "\n".join(
        f"- {t['function']['name']}({', '.join(t['function']['parameters']['properties'])}) "
        f"returns: {self.outputs.get(t['function']['name'], 'a single value')}"
        for t in self.tools
    )
```

The plan itself comes back through `extract()` (chapter 1, notebook 5) against a Pydantic model, so it's validated JSON before anything looks at it:

```python
class PlanStep(BaseModel):
    tool: str
    args: dict[str, str]

class Plan(BaseModel):
    steps: list[PlanStep]
```

Note `args: dict[str, str]` — every argument is a string, including numbers. That's deliberate: `"$step1.temp_f"` and `"100"` have to live in the same field, so the type has to accommodate the reference. The real values get restored when the reference is resolved, and anything still wrong by then is `DebugThink`'s problem.

Let's make a plan and look at it:

In [3]:
planner = PlannerThink(tools=tools, outputs=OUTPUTS)

planned = Conversation()
planned.messages.append({"role": "user", "content": REQUEST})
plan = planner.run(planned)        # the plan, and nothing else
planned.messages.append(plan)

print("kind:", plan["kind"], "| decisions produced:", 0)
for i, step in enumerate(plan["steps"]):
    print(f"step{i}: {step['tool']}({step['args']})")

kind: tools | decisions produced: 0
step0: get_current_time({'timezone': 'Asia/Tokyo'})
step1: get_weather({'city': 'Tokyo'})
step2: convert_temperature({'value': '$step1.temp_f', 'from_unit': 'F', 'to_unit': 'C'})


Three steps, and one of them carries `"$step1.temp_f"` — the planner correctly worked out that it needs a number it can't have yet, and that the number is one field of step 1's object rather than the whole thing. Forwarding the whole JSON object would have failed validation: `ConvertTemperatureArgs.value` is a `float`.

Resolution happens later, mechanically, in `think.py` — one regex shared by everything that
has to turn a reference back into a value:

```python
_STEP_REF = re.compile(r"\$(?:step)?(\d+)(?:\.(\w+))?")

def resolve_args(step, results):
    return {k: _resolve_one(v, results) for k, v in step["args"].items()}
```

A regex and a dictionary lookup. No model call, no judgment — the decision about what to reference was made once, by the planner, at planning time. This is the same division as everywhere else in the chapter: `Think` decides, everything downstream executes what was decided.

And this is what `step_outputs` was for. Notebook 2 introduced it as an indexed list of every raw tool result, deliberately kept out of `messages`. `$step1` indexes straight into it — which is exactly why raw results had to be kept somewhere. The *observation* in `messages` says "The weather is clear, 88.0 degrees Fahrenheit"; you cannot reliably get a float back out of a sentence. `step_outputs[1]` still holds `'{"temp_f": 88.0, "condition": "clear"}'`.

Two of those steps depend on nothing at all. Getting the time in Tokyo and getting the weather in Tokyo have no relationship to each other; running one before the other is an accident of how the text was ordered.

`$stepN` didn't only solve the unknown-argument problem. It **is** the dependency edge — the planner had to state precisely which earlier results each step needs, and having stated it, the graph is already there to be read.

That is also how execution *starts*. There is no first instruction to kick things off: `Graph` looks at every step, and any whose dependency set is empty has nothing to wait for, so it goes immediately:

```python
def launch_ready():
    for i, step in enumerate(steps):
        if i in started or not (deps[i] <= results.keys()):
            continue                      # still waiting on something
        started.add(i)
        pool.submit(self.act.execute, step["tool"], json.dumps(resolve_args(step, results)))

launch_ready()      # <- ignition: everything with no unmet references
```

`resolve_args` rewrites only values matching `$stepN`; a literal the planner already knew, like `"Tokyo"`, passes straight through. So the first calls go out with their arguments fully formed, and each step that finishes releases whatever was waiting on it.


In [4]:
for i, step in enumerate(plan["steps"]):
    deps = step_dependencies(step)
    print(f"step{i}: {step['tool']:20} depends on {sorted(deps) if deps else 'nothing'}")

step0: get_current_time     depends on nothing
step1: get_weather          depends on nothing
step2: convert_temperature  depends on [1]


The plan is written as a list because lists are how you write things down, but it never was one. Two of those steps depend on nothing at all — getting the time in Tokyo and getting the weather in Tokyo have no relationship to each other whatsoever. Running one before the other is an accident of how the text was ordered.

`$stepN` didn't just solve the unknown-argument problem. It **is** the dependency edge — the planner had to state, precisely, which earlier results each step needs, and having stated it, the graph is already there to be read.

`Graph` executes against that structure instead of the ordering:

```python
class Graph:
    def run(self, conversation):
        steps = latest_plan(conversation)["steps"]
        deps = [step_dependencies(step) for step in steps]
        results, started = {}, set()

        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as pool:
            in_flight = {}

            def launch_ready():
                for i, step in enumerate(steps):
                    if i in started or not (deps[i] <= results.keys()):
                        continue
                    started.add(i)
                    args = resolve_args(step, results)
                    in_flight[pool.submit(self.act.execute, step["tool"], json.dumps(args))] = i

            launch_ready()
            while in_flight:
                finished = next(concurrent.futures.as_completed(in_flight))
                i = in_flight.pop(finished)
                effect = finished.result()
                results[i] = effect["content"]
                conversation.messages.append({"role": "tool", "content": self._phrase(effect, conversation)})
                launch_ready()
        return None
```

`launch_ready()` starts every step whose dependencies are all in `results`. Then the loop waits for whichever step finishes *first* — `as_completed` — records it, and calls `launch_ready()` again. So a step begins the instant the specific things it was waiting for are done, regardless of what else is still running.

`resolve_args` reads from the local `results` dict rather than `step_outputs`, because these steps didn't run in index order and `step_outputs` is append-ordered. Both live in `think.py` beside the plan types, so the one regex that understands `$stepN` has a single home -- `resolve_text`, which strips references out of a final answer, uses it too.

Same plan, executed both ways:

In [4]:
act = Act(functions=FUNCTIONS, schemas=SCHEMAS)

# Deterministic extractors, so this measures SCHEDULING and nothing else. Without them `Graph`
# makes an LLM call to phrase every result while the sequential loop below makes none -- and the
# comparison stops being about overlap at all. Run that way against a slower model, the graph
# came out *behind*: 3.9s against 3.0s, three phrasing calls swamping the 1.5s the parallelism
# saved. An unfair benchmark that happens to flatter you is still an unfair benchmark.
EXTRACTORS = {
    "get_current_time": lambda e: f"the time is {e['content']}",
    "get_weather": lambda e: f"the weather is {e['content']}",
    "convert_temperature": lambda e: f"that is {e['content']} degrees Celsius",
}

# one at a time, in the order written
start = time.perf_counter()
results = {}
for i, step in enumerate(plan["steps"]):
    results[i] = act.execute(step["tool"], json.dumps(resolve_args(step, results)))["content"]
one_at_a_time = time.perf_counter() - start

# the same steps, scheduled by dependency
graphed = Conversation()
graphed.messages.append({"role": "user", "content": REQUEST})
graphed.messages.append(plan)

start = time.perf_counter()
Graph(act=act, extractors=EXTRACTORS).run(graphed)
graph_seconds = time.perf_counter() - start

print(f"one at a time: {one_at_a_time:.1f}s")
print(f"Graph:         {graph_seconds:.1f}s")
print()
for m in graphed.messages:
    if m["role"] == "tool":
        print(f"- tool {m['content']!r}")

one at a time: 3.0s
Graph:         1.5s

- tool 'the time is 2026-08-18 06:54 JST'
- tool 'the weather is {"temp_f": 88.0, "condition": "clear"}'
- tool 'that is 31.1 degrees Celsius'


Three seconds against one and a half. The two independent steps overlapped, so the total is the longest path through the graph rather than the sum of every step — and each of them sleeps 1.5s, so the arithmetic is visible: 1.5 + 1.5 sequentially, max(1.5, 1.5) as a graph, with the dependent third step adding nothing measurable.

**The extractors are load-bearing in that measurement, not decoration.** Without them `Graph` phrases every result with an LLM call while the sequential loop above makes none, and the comparison stops being about scheduling. Measured that way against a slower model, the graph came out *behind* — 3.9s against 3.0s — because three phrasing calls cost more than the 1.5s the overlap saved. The parallelism was real the whole time; the benchmark was measuring something else. A benchmark that flatters you is worth distrusting exactly as much as one that doesn't.

The observations still come back in *completion* order, not plan order — direct evidence that they raced rather than queued.

### Threads here, not `asyncio`

Chapter 1's notebook 8 made the case for `asyncio` and proved a real speedup with `gather`. This chapter uses `concurrent.futures.ThreadPoolExecutor` instead, and the reason is practical rather than principled: notebooks already run inside an event loop, and library code that starts its own conflicts with it in ways that are genuinely unpleasant to debug. Threads sidestep that entirely.

Nothing is lost. Every one of these calls is blocking I/O — an HTTP request or a `sleep` — so the thread spends its time waiting, and waiting is exactly what releases the GIL. This is I/O-bound work, which threads handle just as well as coroutines.

### Why `Act.execute()` had to be pure

Notebook 3 mentioned in passing that `execute()` reads and writes nothing shared. Here is where that stops being a stylistic note. `pool.submit(self.act.execute, ...)` runs the *same* `Act` object from several threads at once. Had `execute()` accumulated anything on `self` — a retry counter, a log, the last error — those threads would be writing over each other, and the resulting bug would appear only under concurrency and only sometimes.

`execute()` creates its own `DebugThink` and its own `attempts` list per call, as locals. Every concurrent call gets its own. That was a deliberate choice made two notebooks before anything ran in parallel, and it's why nothing had to change here.

### What `Graph` costs

### What committing to a plan costs

`Graph` executes a plan wholesale, and that is a real trade rather than a free win.

It commits. Every step whose references are satisfied goes out immediately, which is the entire point — and it means a step whose usefulness depended on an earlier result being surprising runs anyway. Nothing looks at a result and reconsiders; that is what the replanning round is for, and it only happens once the whole graph has finished.

It is also the one component in this chapter that imports from `think.py`. `Runtime` and `Loop` know nothing about what they're running; `Graph` needs `latest_plan`, `step_dependencies` and `resolve_args`, because it schedules *plans* specifically. That isn't a leak to fix — it's an honest admission that `Graph` is a plan executor, not a general primitive.


### Running the graph in a loop

`Graph` above ran once, against a plan that was already complete. Put it in a loop with the planner and you get the pattern usually called **LLM Compiler**: plan a round of calls, run them as a graph, then let the planner look at what came back and decide whether to answer or add more steps.

```
Observe(event) -> PlannerThink -> Graph        repeat_from=1
```

Three components, and both of the things that make it work were missing until this loop was actually run — which is the point of running it.

**Deciding to answer is itself an Act.** When the planner concludes "done, here is the answer", that is a `decision`, not a final message; something has to turn it into one. `Graph` does, because a graph of Acts can perfectly well contain exactly one:

```python
last = pending_decision(conversation)
if last is not None and last.get("type") == "respond":
    return self.act.run(conversation)
```

(`pending_decision` — notebook 2 — is "the newest decision nobody has carried out yet". Asking
merely for the newest *decision* would find one `Act` had already executed and run it twice.)

Without it the loop plans, executes, concludes it is finished, and then comes back round to re-run the completed plan until the budget stops it. The agent works out the answer and never says it.

**`$stepN` counts across rounds, not within them.** `Graph` used to resolve references against a dictionary it created fresh on every call, so in round two `$step0` meant *round two's* first step. But `ContinueOrFinish` explicitly tells the planner to keep numbering from where the previous round stopped, and the one-step-at-a-time path resolves against `conversation.step_outputs`, which persists. The two execution paths disagreed about what `$step0` meant, and the graph was the one that was wrong. It now seeds earlier rounds' outputs as already-satisfied dependencies and appends its own by index.

Neither fault could show up while `Graph` only ever ran once.


In [6]:
compiler = Runtime(
    loop=[
        Observe(event=REQUEST),
        PlannerThink(tools=tools, outputs=OUTPUTS),
        Graph(act=Act(functions=FUNCTIONS, schemas=SCHEMAS)),
    ],
    repeat_from=1,
    max_iterations=12,
)

compiled = compiler.run(Conversation())
for m in compiled.messages:
    print(f"- {m['role']:9} {m['content']!r}")
print()
print("step_outputs across every round:", compiled.step_outputs)

- user      "What's the current time in Tokyo, and what's the temperature there in Celsius?"
- plan      '{"steps":[{"tool":"get_current_time","args":{"timezone":"Asia/Tokyo"}},{"tool":"get_weather","args":{"city":"Tokyo"}},{"tool":"convert_temperature","args":{"value":"$step1.temp_f","from_unit":"F","to_unit":"C"}}]}'
- tool      'The current time in Tokyo is 2026-08-18\u202f06:53\u202fJST.'
- tool      'The current time in Tokyo is 2026-08-18\u202f06:53\u202fJST, and the temperature is about\u202f31.1\u202f°C.'
- tool      'The temperature in Tokyo is 31.1\u202f°C.'
- decision  'The current time in Tokyo is 2026-08-18 06:53 JST, and the temperature is 31.1°C.'
- assistant 'The current time in Tokyo is 2026-08-18 06:53 JST, and the temperature is 31.1°C.'

step_outputs across every round: ['2026-08-18 06:53 JST', '{"temp_f": 88.0, "condition": "clear"}', '31.1']


Two rounds. The first planned and executed; the second read the evidence and answered, and that answer came out of `Graph` running a single Act. `step_outputs` holds every round's results in one list, which is what makes a replanned step able to refer back to an earlier round.

**One thing the planner will do if you let it: use `$stepN` in the answer itself.** Run live,
this loop replied

> *"The current time in Tokyo is $step1 and 100 degrees Fahrenheit is equal to $step0 degrees Celsius."*

Every check passed — the run terminated, the last message was an `assistant` message — and the
answer was useless. The planner is taught that syntax so it can chain steps, and it reaches for
it in prose too. So the prompt says not to, and `resolve_text()` substitutes any that slip
through anyway. Telling a model not to do something is necessary; it is not sufficient.

Count the model calls: the planner twice, plus one phrasing call per observation, and nothing in between. However many steps a plan has, the model is consulted once to write it and once to judge the results.

Strip the loop and this is **ReWOO** — same components, unrolled to exactly two rounds, unable to go round again if the first plan turns out to be wrong. Notebook 6 puts the two side by side.


## A plan of goals, not calls

Everything above plans *tool calls*. The planner decides `get_weather(city="Tokyo")` before anything runs, and `Graph` executes the lot without asking anybody. That works because the tools are simple enough to see all the way through from the start.

Now suppose a step is "find the three largest cities in Japan." There is no single call that does it, and which calls it takes depends on what the first one returns. A planner that must commit to exact calls up front cannot express that step at all.

So the other kind of plan says what to *achieve* and leaves the how open:

```python
class Goals(BaseModel):
    goals: list[str]
```

`PlannerThink(goals=True)` writes those and stops. Two things it deliberately does **not** do.

**It emits no decision.** The tool-call planner returns the first step's `tool_call` for `Act` to execute. A goal is not executable — working out how to reach one is reasoning, and reasoning belongs to a `Think`. So the goal planner returns only the plan itself, and control passes to whatever component comes next.

**It marks the plan `kind="goals"`, which keeps the mechanical machinery away from it.** `latest_plan` filters on `kind="tools"`, so `Graph` will never try to execute a goal as though it were a call. Two plan shapes, one message type, no collisions:

| | tool-call plan | goal plan |
|---|---|---|
| a step is | `{"tool": ..., "args": {...}}` | a sentence |
| who executes it | `Graph`, mechanically | a `Think`, by deciding |
| emits a decision | no — `Graph` reads the plan itself | no |
| `kind` | `"tools"` | `"goals"` |


In [7]:
goal_planner = PlannerThink(tools=tools, outputs=OUTPUTS, goals=True)

asking = Conversation()
asking.messages.append({"role": "user", "content": REQUEST})
goal_plan = goal_planner.run(asking)

print("kind:", goal_plan["kind"], "| emits a decision:", goal_plan["role"] == "decision")
for i, step in enumerate(goal_plan["steps"]):
    print(f"  goal {i}: {step['goal']}")

kind: goals | emits a decision: False
  goal 0: Obtain the current local time in Tokyo.
  goal 1: Obtain the current temperature in Tokyo in Celsius.


No tool is named anywhere in those goals, which is the point — they say what a good answer needs, not how to get it.

Now put the planner in a loop with an ordinary `Think`. The planner runs once at position 1; `repeat_from=2` cycles `Think → Act → Observe` after it, which is notebook 2's ReAct loop reading a set of goals instead of a bare request:

In [8]:
goals_first = Runtime(
    loop=[
        Observe(event=REQUEST),
        PlannerThink(tools=tools, outputs=OUTPUTS, goals=True),   # writes goals, decides nothing
        Think(tools=tools, allow_fork=False),                     # works out how, one step at a time
        Act(functions=FUNCTIONS, schemas=SCHEMAS),
        Observe(),
    ],
    repeat_from=2,
    max_iterations=16,
)

done = goals_first.run(Conversation())
for m in done.messages:
    print(f"- {m['role']:9} {m['content']!r}")
    if m.get("reasoning"):
        print(f"             why: {m['reasoning']}")

- user      "What's the current time in Tokyo, and what's the temperature there in Celsius?"
- plan      'Goals:\n0. Obtain the current local time in Tokyo.\n1. Obtain the current temperature in Tokyo in Celsius.'
- decision  '{"timezone":"Asia/Tokyo"}'
- tool      'The current time in Tokyo is 06:53\u202fJST on 18\u202fAugust\u202f2026.'
- decision  '{"city":"Tokyo"}'
- tool      'The current temperature in Tokyo is approximately 31\u202f°C.'
- decision  'The current time in Tokyo is 06:53\u202fJST on 18\u202fAugust\u202f2026, and the temperature is about 31\u202f°C.'
- assistant 'The current time in Tokyo is 06:53\u202fJST on 18\u202fAugust\u202f2026, and the temperature is about 31\u202f°C.'


The planner set the agenda; the `Think` chose the calls, one at a time, reading each result before picking the next. Nothing in the plan mentioned a tool.

That difference is what separates the named patterns, and it is smaller than the names suggest:

- **Plan & Execute** — plan of *goals*, each worked out by an agent that can look at results as it goes. That is the loop above.
- **ReWOO** — *Reasoning WithOut Observation*. Plan of *exact calls* with `$stepN` placeholders, executed without consulting the model again, then one final pass to write the answer. The model is called twice no matter how many steps there are. Cheapest, and blind: nothing notices a bad plan until the end. That is the sequential loop earlier in this notebook.
- **LLM Compiler** — ReWOO's up-front call plan, but run as a graph so independent steps overlap, plus a step at the end that can send it back for replanning. That is `Graph` plus `ContinueOrFinish`.

So the three differ on two axes only: whether a step names a call or a goal, and whether anything looks at results before the end. Everything else they share.

### Revising the plan while carrying it out

The planner runs once, at position 1, and `repeat_from=2` steps straight over it — so nothing
above can change the goals once they are written. That is usually right, and occasionally wrong:
executing a plan is exactly when you discover it was the wrong plan.

The `Think` doing the executing is the only thing that finds that out, so it gets a `replan` tool
— offered only once a plan of goals exists, and only to a `Think` declared with
`allow_replan=True`:

```python
Think(tools=tools, allow_fork=False, allow_replan=True)
```

It returns a **plan**, not a decision, exactly as `PlannerThink` does — there is nothing for
`Act` to execute, because a goal is not executable. And it supersedes rather than merges:
`latest_plan` already returns the newest, so no other component has to know a revision happened.
The old plan stays in the transcript as a record of what was tried.

Two details, both learned by getting them wrong.

**It asks for the whole plan, not a delta.** A delta needs an edit language — replace goal 2,
insert after goal 1 — and indices that shift as edits apply. Whole-plan replacement makes the
merge logic *"`latest_plan` returns the newest"* and nothing else. The cost is that a model
restating eight goals may quietly reword one, so the guidance is explicit: copy untouched goals
word for word, and put new ones where they belong in the order.

**That guidance lives in the system message, not the tool schema.** The first version explained
it all inside `replan`'s own description and embedded the current plan — 1202 characters, twice
the size of the two real tools combined. Offered alongside them, the model then failed to emit
*any* valid tool call, five attempts running. Remove `replan` and the same request succeeded
first try. Tool schemas are a shared budget: `replan` did not break `replan`, it broke
`get_current_time`.

One thing the loop above does *not* do is delegate. Two of these goals are independent and could run at the same time — but noticing that and splitting the work is a separate judgment, and it belongs to `Think`, not to the planner. Notebook 5 gives that same `Think` the ability to fork, and the identical goal plan can then be farmed out to sub-agents instead of worked through in sequence. The plan does not change; only what the `Think` after it decides to do with it.

## What you built

✓ Built `PlannerThink`, producing an entire validated sequence of tool calls in one `extract()` call instead of one decision per turn

✓ Solved the unknown-argument problem with `$stepN` and `$stepN.field` references, and an `outputs` dict describing what each tool returns — since tool schemas describe arguments and never results

✓ Found that those references *are* the dependency graph, and that a step with no references has nothing to wait for — which is exactly what lets `Graph` start it, with no instruction needed to begin

✓ Ran the same plan step-by-step and as a graph, and measured the difference — observations arriving in completion order rather than plan order

✓ Ran the full compiler loop, plan → graph → replan → answer, and found two faults that only a second round could expose: nothing to turn the final decision into an answer, and `$stepN` restarting its count each round

✓ Built the other kind of plan — `PlannerThink(goals=True)`, which writes what to achieve rather than which calls to make, emits no decision because a goal is not executable, and is tagged `kind="goals"` so the plan machinery leaves it alone — then ran it with an ordinary `Think` working out the calls itself

✓ Placed Plan & Execute, ReWOO and LLM Compiler against each other on the only two axes that separate them: whether a step names a call or a goal, and what the third role is allowed to do

**Next:** `05_sub_agents.ipynb` — splitting the agent rather than the plan: delegating to sub-agents that run their own loops, having them debate each other, and choosing between what they bring back.
